# PanVision — P3 nnU-Net (pancreas)

nnU-Net v2 `2d` on the **same** train/val patients as P1/P2, the same preprocessed volumes, and scored with the same
`evaluate.py`. Shortened schedule: 50 epochs × 250 iterations (`nnUNetTrainer_PanVision`, checkpoint every 2 epochs,
no mirroring — same as the U-Nets). Expect several hours on a free T4, possibly across sessions.

**Before the first run:** open the shared Drive link → right-click the folder → *Organize → Add shortcut* → *My Drive*.
Then **Runtime → Change runtime type → T4 GPU**.

Free Colab disconnects after a few hours. Nothing is lost: models, logs and results are saved to the shared Drive
folder. After a disconnect, run **Setup** again, then re-run the cell that was interrupted — training resumes.

## 1. Setup (run every session)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
%cd /content
# Leave the repo folder first, so re-running this cell is safe.
from google.colab import drive
drive.mount('/content/drive')
# Private repo: clone with a GitHub token stored as a Colab secret named GITHUB_TOKEN (key icon in the left bar).
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
# Update in place instead of deleting the folder, so a training run in the background keeps working.
if os.path.exists('/content/Panvision/.git'):
    !git -C /content/Panvision fetch -q https://{token}@github.com/hrihaanG/Panvision.git segmentation && git -C /content/Panvision reset -q --hard FETCH_HEAD
else:
    !git clone -q -b segmentation https://{token}@github.com/hrihaanG/Panvision.git /content/Panvision
!git -C /content/Panvision remote set-url origin https://github.com/hrihaanG/Panvision.git  # drop the token from git config
%cd /content/Panvision
assert os.path.exists('src/segmentation'), 'Clone failed: check the GITHUB_TOKEN secret and its notebook access'
!pip -q install monai SimpleITK nnunetv2

In [ ]:
import os, glob
hits = sorted(set(p[:-len('/processed/full')] for p in glob.glob('/content/drive/MyDrive/**/processed/full', recursive=True)))
print('\n'.join(hits) or 'Not found: add the shared folder as a shortcut in My Drive (see top).')
# The shared folder name contains spaces, which break shell commands: use a space-free link to it.
if not os.path.exists('/content/data'):
    os.symlink(hits[0], '/content/data')  # change hits[0] if more than one folder is listed
DATA = '/content/data'
os.environ['PANVISION_DATA_DIR'] = DATA
CACHE = '/content/cache'
M = f'{DATA}/models/segmentation'
R = f'{DATA}/outputs/segmentation/results'
NNUNET_PRED = f'{DATA}/outputs/segmentation/nnunet/predictions'
print('Using', DATA)

In [ ]:
NN = f'{M}/P3_nnunet_2d'
os.makedirs(NN, exist_ok=True)
os.environ.update(nnUNet_raw='/content/nnunet/raw', nnUNet_preprocessed='/content/nnunet/preprocessed',
                  nnUNet_results=f'{NN}/results', nnUNet_n_proc_DA='2')
!python -m src.segmentation.nnunet_export install-trainer

## 2. nnU-Net preprocessing (first session ~30 min; later sessions restore it from Drive)

In [ ]:
if os.path.exists(f'{NN}/preprocessed.tar'):
    !mkdir -p /content/nnunet && tar -xf "{NN}/preprocessed.tar" -C /content/nnunet
else:
    !python -m src.segmentation.nnunet_export dataset --raw $nnUNet_raw
    !nnUNetv2_plan_and_preprocess -d 501 -c 2d -np 2 --verify_dataset_integrity
    !python -m src.segmentation.nnunet_export splits --raw $nnUNet_raw --preprocessed $nnUNet_preprocessed
    !tar -cf "{NN}/preprocessed.tar" -C /content/nnunet preprocessed
!cat /content/nnunet/preprocessed/Dataset501_PanVisionPancreas/splits_final.json | head -c 300

## 3. Train (~5–8 h on a free T4, possibly across sessions)

In [ ]:
# Runs as a background job: a browser disconnect does not stop it. After reconnecting, run the setup cells and
# this cell again to see progress (after a runtime reset it resumes from the last checkpoint via --c).
# Interrupting this cell only stops watching; stop('P3_nnunet_2d') ends training.
from src.segmentation.colab_jobs import follow, launch, stop
assert launch('P3_nnunet_2d', 'nnUNetv2_train 501 2d 0 -tr nnUNetTrainer_PanVision --c'), 'nnU-Net training did not finish'


## 4. Predict val/test cases and score on val

In [ ]:
!python -m src.segmentation.nnunet_export inputs --out /content/nnunet/inputs
!nnUNetv2_predict -i /content/nnunet/inputs -o "{NNUNET_PRED}" -d 501 -c 2d -f 0 -tr nnUNetTrainer_PanVision -chk checkpoint_best.pth -npp 1 -nps 1
!python -m src.segmentation.evaluate pancreas --predictions "{NNUNET_PRED}" --name P3_nnunet_2d --split val --max-studies 100

Now go back to `colab_segmentation.ipynb` → *Compare pancreas candidates*; P3 is picked up automatically.